In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.chat_models import init_chat_model

In [2]:
class OutputGuardrailMiddleware(AgentMiddleware):
    def after_agent(self, state, runtime):
        last_message = state["messages"][-1]
        response = last_message.content

        sensitive_words = [
            "password",
            "api key",
            "secret token",
            "database credentials"
        ]
        for word in sensitive_words:
            if word.lower() in response.lower():
                last_message.content = (
                    "Sensitive information detected and removed from the response."
                )
                return None

        harmful_words = [
            "malware",
            "hack system",
            "illegal access"
        ]
        for word in harmful_words:
            if word.lower() in response.lower():
                last_message.content = (
                    "Unsafe content detected and blocked."
                )
                return None

        MAX_LENGTH = 500
        if len(response) > MAX_LENGTH:
            last_message.content = (
                response[:MAX_LENGTH]
                + "\n\n[Response truncated]"
            )

        if not response.endswith("."):
            last_message.content += "."
        return None

In [3]:
llm = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [4]:
agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        OutputGuardrailMiddleware()
    ]
)

In [5]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Tell me about hacking systems"
        }
    ]
})

In [6]:
print(response["messages"][-1].content)

Sensitive information detected and removed from the response.
